
# 11 - OLS feature blocks

This notebook studies the first OLS diagnostic axis:

> Which feature families add linear predictive signal?

It compares the ordered feature-block experiments:

1. `ols_demo`
2. `ols_demo_educ`
3. `ols_demo_educ_labor`
4. `ols_core`
5. `ols_core_no_pyramid`

The goal is not to pick a winner, but to quantify how much signal is added by demographic, education, labor, household/housing, and household-pyramid feature families.

Expected backend layout:

```text
reports/runs/<run_id>/
  metrics/model_comparison.csv
  predictions/test_predictions.parquet
  predictions/validation_predictions.parquet
  diagnostics/distribution_compression_summary.csv
  diagnostics/error_by_income_decile.csv
  feature_columns.json
  dataset_card.json
  config_used.yaml
```



## 00. Setup and run discovery

This section is intentionally coupled to the backend artifact layout. The notebook reads completed backend runs and does not train models.


In [ ]:

from pathlib import Path
import json
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Robust root resolution: works from repo root, notebooks/, or a copied notebook.
ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/home/matias/repos/income-modeling-eph"),
]

ROOT = next(
    (p for p in ROOT_CANDIDATES if (p / "reports" / "runs").exists()),
    Path("/home/matias/repos/income-modeling-eph"),
)

RUNS_DIR = ROOT / "reports" / "runs"
DATASET_PATH = ROOT / "data" / "processed" / "modeling_dataset.parquet"

OUTPUT_DIR = ROOT / "reports" / "notebook_outputs" / "ols_feature_blocks"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

print("ROOT:", ROOT)
print("RUNS_DIR:", RUNS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


def latest_run(pattern: str) -> Path | None:
    candidates = sorted(RUNS_DIR.glob(pattern), key=lambda p: p.name)
    if not candidates:
        return None
    return candidates[-1]


def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def read_parquet_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_parquet(path)


def read_json_if_exists(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text())


def read_yaml_if_exists(path: Path):
    if not path.exists():
        return None
    return yaml.safe_load(path.read_text())


RUN_PATTERNS = {
    "ols_demo": "ols_demo_*",
    "ols_demo_educ": "ols_demo_educ_*",
    "ols_demo_educ_labor": "ols_demo_educ_labor_*",
    "ols_core": "ols_core_*",
    "ols_core_no_pyramid": "ols_core_no_pyramid_*",
}

# Guard against overly broad patterns. Longer names must be resolved before shorter names.
# Example: ols_core_* would also match ols_core_no_pyramid_* if we are careless.
EXACT_EXPERIMENT_IDS = list(RUN_PATTERNS.keys())


def discover_ols_runs() -> dict[str, Path]:
    runs = {}
    for experiment, pattern in RUN_PATTERNS.items():
        candidates = sorted(RUNS_DIR.glob(pattern), key=lambda p: p.name)
        if not candidates:
            runs[experiment] = None
            continue

        # Prefer runs whose config_used.yaml has exactly the expected experiment id.
        exact = []
        for candidate in candidates:
            config = read_yaml_if_exists(candidate / "config_used.yaml") or {}
            experiment_id = ((config.get("experiment") or {}).get("id"))
            if experiment_id == experiment:
                exact.append(candidate)

        runs[experiment] = exact[-1] if exact else candidates[-1]
    return runs


RUNS = discover_ols_runs()

run_table = pd.DataFrame(
    [
        {
            "experiment": exp,
            "run_dir": str(path) if path else None,
            "found": path is not None,
        }
        for exp, path in RUNS.items()
    ]
)

run_table



## 01. Load canonical backend artifacts

The notebook builds a compact run-level table from metrics, predictions, config snapshots, and feature-column metadata.


In [ ]:

ORDER = [
    "ols_demo",
    "ols_demo_educ",
    "ols_demo_educ_labor",
    "ols_core",
    "ols_core_no_pyramid",
]

DISPLAY_LABELS = {
    "ols_demo": "Demographic",
    "ols_demo_educ": "+ Education",
    "ols_demo_educ_labor": "+ Labor",
    "ols_core": "+ Household/housing + pyramid",
    "ols_core_no_pyramid": "Core without pyramid",
}

def load_model_comparison(run_dir: Path, experiment: str) -> pd.DataFrame:
    df = read_csv_if_exists(run_dir / "metrics" / "model_comparison.csv")
    if df.empty:
        return df
    df["experiment"] = experiment
    df["run_dir"] = str(run_dir)
    return df


def load_predictions(run_dir: Path, experiment: str, split: str) -> pd.DataFrame:
    path = run_dir / "predictions" / f"{split}_predictions.parquet"
    df = read_parquet_if_exists(path)
    if df.empty:
        return df

    df["experiment"] = experiment
    df["run_dir"] = str(run_dir)
    df["split"] = split

    if "residual" not in df.columns:
        df["residual"] = df["y_true"] - df["y_pred"]
    if "abs_error" not in df.columns:
        df["abs_error"] = df["residual"].abs()
    if "squared_error" not in df.columns:
        df["squared_error"] = df["residual"] ** 2

    return df


def load_run_metadata(run_dir: Path, experiment: str) -> dict:
    feature_columns = read_json_if_exists(run_dir / "feature_columns.json") or []
    dataset_card = read_json_if_exists(run_dir / "dataset_card.json") or {}
    config_used = read_yaml_if_exists(run_dir / "config_used.yaml") or {}
    feature_view = config_used.get("feature_view") or {}

    return {
        "experiment": experiment,
        "run_dir": str(run_dir),
        "n_feature_columns": len(feature_columns),
        "feature_columns": feature_columns,
        "feature_view_name": feature_view.get("name"),
        "include_blocks": feature_view.get("include_blocks"),
        "dataset_card": dataset_card,
        "config_used": config_used,
    }


model_comparison_parts = []
prediction_parts = []
metadata_rows = []

for experiment, run_dir in RUNS.items():
    if run_dir is None:
        continue

    model_comparison_parts.append(load_model_comparison(run_dir, experiment))
    for split in ["validation", "test"]:
        prediction_parts.append(load_predictions(run_dir, experiment, split))
    metadata_rows.append(load_run_metadata(run_dir, experiment))

model_comparison = (
    pd.concat([d for d in model_comparison_parts if not d.empty], ignore_index=True)
    if any(not d.empty for d in model_comparison_parts)
    else pd.DataFrame()
)

predictions = (
    pd.concat([d for d in prediction_parts if not d.empty], ignore_index=True)
    if any(not d.empty for d in prediction_parts)
    else pd.DataFrame()
)

run_metadata = pd.DataFrame(metadata_rows)

print("model_comparison:", model_comparison.shape)
print("predictions:", predictions.shape)
print("run_metadata:", run_metadata.shape)

run_metadata[["experiment", "feature_view_name", "include_blocks", "n_feature_columns"]]



## 02. Build analysis tables

The core table is one row per experiment and split, with aggregate error metrics and prediction-compression diagnostics.


In [ ]:

def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    if len(y_true) == 0:
        return np.nan
    denom = np.sum((y_true - y_true.mean()) ** 2)
    if denom == 0:
        return np.nan
    return 1 - np.sum((y_true - y_pred) ** 2) / denom


def aggregate_prediction_metrics(g):
    y = g["y_true"].astype(float)
    yhat = g["y_pred"].astype(float)
    residual = y - yhat
    return pd.Series(
        {
            "n": len(g),
            "r2": safe_r2(y, yhat),
            "mae": residual.abs().mean(),
            "rmse": np.sqrt((residual ** 2).mean()),
            "mean_error": (yhat - y).mean(),
            "sd_y_true": y.std(),
            "sd_y_pred": yhat.std(),
            "compression_ratio": yhat.std() / y.std() if y.std() else np.nan,
            "p95_abs_error": residual.abs().quantile(0.95),
        }
    )


if predictions.empty:
    raise RuntimeError(
        "No predictions were loaded. Run the OLS feature-block experiments first "
        "or check RUN_PATTERNS."
    )

metrics_by_prediction = (
    predictions
    .groupby(["experiment", "split"], as_index=False)
    .apply(aggregate_prediction_metrics, include_groups=False)
    .reset_index(drop=True)
)

metrics_by_prediction["experiment_order"] = metrics_by_prediction["experiment"].map(
    {exp: i for i, exp in enumerate(ORDER)}
)
metrics_by_prediction["label"] = metrics_by_prediction["experiment"].map(DISPLAY_LABELS)

# Main validation table.
validation_summary = (
    metrics_by_prediction
    .query("split == 'validation'")
    .sort_values("experiment_order")
    [[
        "experiment",
        "label",
        "n",
        "r2",
        "mae",
        "rmse",
        "compression_ratio",
        "sd_y_true",
        "sd_y_pred",
        "mean_error",
        "p95_abs_error",
    ]]
    .reset_index(drop=True)
)

# Incremental changes relative to previous row.
for col in ["r2", "mae", "rmse", "compression_ratio"]:
    validation_summary[f"delta_{col}_vs_previous"] = validation_summary[col].diff()

# Changes relative to ols_core.
core_row = validation_summary[validation_summary["experiment"] == "ols_core"]
if not core_row.empty:
    core_metrics = core_row.iloc[0]
    for col in ["r2", "mae", "rmse", "compression_ratio"]:
        validation_summary[f"delta_{col}_vs_core"] = validation_summary[col] - core_metrics[col]

TABLE_DIR.mkdir(parents=True, exist_ok=True)
validation_summary.to_csv(TABLE_DIR / "T1_ols_feature_blocks_validation_summary.csv", index=False)
metrics_by_prediction.to_csv(TABLE_DIR / "T2_ols_feature_blocks_metrics_by_split.csv", index=False)
run_metadata.to_csv(TABLE_DIR / "T3_ols_feature_blocks_run_metadata.csv", index=False)

validation_summary.round(4)



## 03. Main table: validation metrics

This table is the primary evidence for feature-family accumulation. Focus on the validation split.


In [ ]:

display_cols = [
    "experiment",
    "label",
    "n",
    "r2",
    "delta_r2_vs_previous",
    "mae",
    "delta_mae_vs_previous",
    "rmse",
    "compression_ratio",
]

validation_summary[display_cols].round(4)



## 04. Figure 1 — incremental validation R²

This figure shows whether each added feature family raises the linear predictive frontier.


In [ ]:

plot_df = validation_summary.query("experiment != 'ols_core_no_pyramid'").copy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(plot_df["label"], plot_df["r2"], marker="o")
ax.set_title("OLS validation R² by accumulated feature block")
ax.set_xlabel("Feature set")
ax.set_ylabel("Validation R²")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "F1_ols_feature_blocks_r2_incremental.png", dpi=160)
plt.show()



## 05. Figure 2 — MAE and RMSE by feature block

If R² improves but MAE barely moves, the model may be improving variance explained without changing typical absolute errors much.


In [ ]:

plot_df = validation_summary.copy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(plot_df["label"], plot_df["mae"], marker="o", label="MAE")
ax.plot(plot_df["label"], plot_df["rmse"], marker="o", label="RMSE")
ax.set_title("OLS validation error by feature block")
ax.set_xlabel("Feature set")
ax.set_ylabel("Error in log-income units")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "F2_ols_feature_blocks_mae_rmse.png", dpi=160)
plt.show()



## 06. Figure 3 — prediction compression

The compression ratio is:

\[
\frac{SD(\hat y)}{SD(y)}
\]

Values below one mean that predictions are less dispersed than observed income.


In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(validation_summary["label"], validation_summary["compression_ratio"])
ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_title("OLS prediction compression by feature block")
ax.set_xlabel("Feature set")
ax.set_ylabel("SD(y_pred) / SD(y_true)")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "F3_ols_feature_blocks_compression_ratio.png", dpi=160)
plt.show()



## 07. Household pyramid sensitivity

`ols_core_no_pyramid` is not part of the accumulation path. It tests whether the household-age-composition block materially changes the OLS benchmark.


In [ ]:

pyramid_compare = (
    validation_summary
    .query("experiment in ['ols_core', 'ols_core_no_pyramid']")
    [[
        "experiment",
        "label",
        "r2",
        "mae",
        "rmse",
        "compression_ratio",
        "sd_y_pred",
    ]]
    .copy()
)

if set(pyramid_compare["experiment"]) == {"ols_core", "ols_core_no_pyramid"}:
    wide = pyramid_compare.set_index("experiment")
    delta = wide.loc["ols_core"][[ "r2", "mae", "rmse", "compression_ratio", "sd_y_pred"]] - wide.loc["ols_core_no_pyramid"][[ "r2", "mae", "rmse", "compression_ratio", "sd_y_pred"]]
    pyramid_delta = delta.rename("core_minus_no_pyramid").to_frame()
else:
    pyramid_delta = pd.DataFrame()

pyramid_compare.round(4), pyramid_delta.round(4)



## 08. Split stability

A feature-block gain is more credible if the pattern is similar across validation and test.


In [ ]:

split_stability = (
    metrics_by_prediction
    .pivot_table(
        index=["experiment", "label", "experiment_order"],
        columns="split",
        values=["r2", "mae", "rmse", "compression_ratio"],
        aggfunc="first",
    )
)

split_stability.columns = [f"{metric}_{split}" for metric, split in split_stability.columns]
split_stability = split_stability.reset_index().sort_values("experiment_order")

if {"r2_validation", "r2_test"}.issubset(split_stability.columns):
    split_stability["r2_test_minus_validation"] = split_stability["r2_test"] - split_stability["r2_validation"]
if {"mae_validation", "mae_test"}.issubset(split_stability.columns):
    split_stability["mae_test_minus_validation"] = split_stability["mae_test"] - split_stability["mae_validation"]

split_stability.to_csv(TABLE_DIR / "T4_ols_feature_blocks_split_stability.csv", index=False)
split_stability.round(4)



## 09. Optional: decile errors

This section uses predictions directly, so it works even if the backend decile diagnostic is missing.


In [ ]:

def assign_deciles_within_split(df):
    result = df.copy()
    result["income_decile"] = (
        result.groupby("split")["y_true"]
        .transform(lambda s: pd.qcut(s, 10, labels=False, duplicates="drop") + 1)
    )
    return result

pred_dec = assign_deciles_within_split(predictions)

decile_errors = (
    pred_dec
    .groupby(["experiment", "split", "income_decile"], as_index=False)
    .agg(
        n=("row_id", "size") if "row_id" in pred_dec.columns else ("y_true", "size"),
        mean_y_true=("y_true", "mean"),
        mean_y_pred=("y_pred", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_error", "mean"),
        rmse=("squared_error", lambda s: np.sqrt(np.mean(s))),
    )
)

decile_errors["experiment_order"] = decile_errors["experiment"].map({exp: i for i, exp in enumerate(ORDER)})
decile_errors["label"] = decile_errors["experiment"].map(DISPLAY_LABELS)
decile_errors.to_csv(TABLE_DIR / "T5_ols_feature_blocks_decile_errors.csv", index=False)

validation_deciles = decile_errors.query("split == 'validation'").copy()

fig, ax = plt.subplots(figsize=(8, 5))
for experiment in ORDER:
    tmp = validation_deciles.query("experiment == @experiment").sort_values("income_decile")
    if tmp.empty:
        continue
    ax.plot(tmp["income_decile"], tmp["mae"], marker="o", label=DISPLAY_LABELS.get(experiment, experiment))

ax.set_title("Validation MAE by income decile")
ax.set_xlabel("Income decile by y_true")
ax.set_ylabel("MAE")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "F4_ols_feature_blocks_decile_mae.png", dpi=160)
plt.show()

validation_deciles.head()



## 10. Automatic diagnostic summary

This is deliberately conservative. Treat it as a notebook aide, not final thesis prose.


In [ ]:

def fmt(x, digits=4):
    if pd.isna(x):
        return "NA"
    return f"{x:.{digits}f}"

summary = validation_summary.set_index("experiment")

diagnostic_notes = []

if {"ols_demo", "ols_demo_educ"}.issubset(summary.index):
    diagnostic_notes.append(
        f"Education increment: ΔR² = {fmt(summary.loc['ols_demo_educ', 'r2'] - summary.loc['ols_demo', 'r2'])}; "
        f"ΔMAE = {fmt(summary.loc['ols_demo_educ', 'mae'] - summary.loc['ols_demo', 'mae'])}."
    )

if {"ols_demo_educ", "ols_demo_educ_labor"}.issubset(summary.index):
    diagnostic_notes.append(
        f"Labor increment: ΔR² = {fmt(summary.loc['ols_demo_educ_labor', 'r2'] - summary.loc['ols_demo_educ', 'r2'])}; "
        f"ΔMAE = {fmt(summary.loc['ols_demo_educ_labor', 'mae'] - summary.loc['ols_demo_educ', 'mae'])}."
    )

if {"ols_demo_educ_labor", "ols_core"}.issubset(summary.index):
    diagnostic_notes.append(
        f"Household/housing+pyramid increment: ΔR² = {fmt(summary.loc['ols_core', 'r2'] - summary.loc['ols_demo_educ_labor', 'r2'])}; "
        f"ΔMAE = {fmt(summary.loc['ols_core', 'mae'] - summary.loc['ols_demo_educ_labor', 'mae'])}."
    )

if {"ols_core", "ols_core_no_pyramid"}.issubset(summary.index):
    diagnostic_notes.append(
        f"Household pyramid sensitivity: core - no_pyramid ΔR² = {fmt(summary.loc['ols_core', 'r2'] - summary.loc['ols_core_no_pyramid', 'r2'])}; "
        f"ΔMAE = {fmt(summary.loc['ols_core', 'mae'] - summary.loc['ols_core_no_pyramid', 'mae'])}."
    )

if "ols_core" in summary.index:
    diagnostic_notes.append(
        f"Core OLS compression ratio: {fmt(summary.loc['ols_core', 'compression_ratio'])}. "
        "Values below 1 indicate prediction compression relative to observed log-income."
    )

DIAGNOSTIC_SUMMARY = "\n".join(f"- {note}" for note in diagnostic_notes)

print(DIAGNOSTIC_SUMMARY)

(TABLE_DIR / "T6_ols_feature_blocks_diagnostic_notes.txt").write_text(DIAGNOSTIC_SUMMARY, encoding="utf-8")



## 11. Outputs written

Tables:

```text
reports/notebook_outputs/ols_feature_blocks/tables/
```

Figures:

```text
reports/notebook_outputs/ols_feature_blocks/figures/
```

The key thesis candidates are:

```text
T1_ols_feature_blocks_validation_summary.csv
F1_ols_feature_blocks_r2_incremental.png
F2_ols_feature_blocks_mae_rmse.png
F3_ols_feature_blocks_compression_ratio.png
F4_ols_feature_blocks_decile_mae.png
```
